# recite

> A passage from its display lines to one audio file with `[start, end]` and word timings per line.

In [ ]:
#| default_exp recite

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| export
import re, json, numpy as np
from pathlib import Path
from fastcore.utils import *
from fastcore.script import call_parse
from vachaka.core import clean, n_syll, concat, Chanter
from vachaka.check import best_of

## Pieces

Display lines are grouped into render pieces. A piece closes after a word holding a daṇḍa, so the two pādas of a hemistich printed as two lines
render as one piece, as Vāgdhenu expects. Prose closes too at `max_syll` akṣaras; verse closes before a line that would pass `max_verse`. Each piece
keeps `(line, word)` for every word, so timings can go back to the lines; a word that cleans to nothing (a daṇḍa, a verse number) stays, untimed.

In [ ]:
#| export
_DANDA = re.compile('[।॥|]')
def word_text(w): return clean(w).strip(' ।॥')

def pieces(lines, prose=False, max_syll=24, max_verse=44):
    "Group display `lines` into pieces, each a list of `(line_index, word)`."
    out, cur, cs = [], [], 0
    def close():
        nonlocal cur, cs
        if any(word_text(w) for _,w in cur): out.append(cur)
        elif out: out[-1] += cur
        cur, cs = [], 0
    for i,l in enumerate(lines):
        ln = n_syll(clean(l))
        if not prose and cur and cs+ln > max_verse: close()
        for w in l.split():
            n = n_syll(word_text(w))
            if prose and cur and cs+n > max_syll: close()
            cur.append((i, w)); cs += n
            if _DANDA.search(w): close()
    if cur: close()
    return out

In [ ]:
ls = ['नित्यानन्दकरी वराभयकरी सौन्दर्यरत्नाकरी', 'निर्धूताखिलघोरपावनकरी प्रत्यक्षमाहेश्वरी ।', 'प्रालेयाचलवंशपावनकरी काशीपुराधीश्वरी', 'भिक्षां देहि कृपावलम्बनकरी मातान्नपूर्णेश्वरी ॥ १ ॥']
ps = pieces(ls)
test_eq(len(ps), 2)
test_eq({i for i,_ in ps[0]}, {0, 1})
test_eq(ps[1][-1], (3, '॥'))
test_eq(len(pieces(['उपदेशं व्याख्यास्यामः॥'], prose=True)), 1)
assert all(sum(n_syll(word_text(w)) for _,w in p) <= 24 for p in pieces(['क '*60], prose=True))

In [ ]:
#| export
def piece_text(p): return ' '.join(o for o in (word_text(w) for _,w in p) if o)

## Lines from words

Word timings come back per piece; each line's span runs from its first timed word to its last.

In [ ]:
#| export
def line_times(n, ps, words):
    "Per line, `dict(s, e, w)` from the pieces' word timings; None for a line with no words."
    out = [None]*n
    for p, ws in zip(ps, words):
        for (i,w), (_, s, e) in zip(p, ws):
            d = out[i] = out[i] or dict(s=None, e=None, w=[])
            d['w'].append([w, s, e])
            if word_text(w): d['s'] = s if d['s'] is None else min(d['s'], s); d['e'] = e if d['e'] is None else max(d['e'], e)
    for d in out:
        if d and d['s'] is None: d['s'] = d['e'] = d['w'][0][1]
    return out

In [ ]:
ps = [[(0,'क'), (0,'ख'), (1,'ग')], [(1,'॥')]]
lt = line_times(3, ps, [[['क',0,100], ['ख',100,200], ['ग',300,400]], [['॥',400,400]]])
test_eq([lt[0]['s'], lt[0]['e'], lt[1]['s'], lt[1]['e']], [0, 200, 300, 400])
test_eq(lt[2], None)

## Reciting sections

A section is recited in parts, each in one voice. By default the section is split into verses, each closing on a line with a double daṇḍa, and
[ganapati](https://github.com/vedicreader/ganapati) scans each: it applies the śloka's rules (Vāgdhenu's own detector passed a 32-syllable prose
sentence as anuṣṭubh) and knows the āryā family. A metre in Vāgdhenu's bank is chanted against its reference, another vṛtta against the fallback
(vasantatilakā), and an āryā-family metre or none is read. A caller can pass its own parts, `dict(lines=[…], meter=…)`, and `kind` forces one voice
for the whole section. Each part takes the best of `tries` seeds by Su-śrotā's error rate.

In [ ]:
#| export
MATRA = {'āryā', 'gīti', 'upagīti', 'udgīti', 'āryāgīti'}
def part_voice(ch, meter):
    "`(kind, ref)` for a part in metre `meter` (None for prose)."
    if not meter or meter in MATRA: return 'read', 'anuṣṭubh'
    return 'chant', ch.bank_ref(meter) or 'vasantatilaka'

In [ ]:
#| export
def verses(lines):
    "Group line indices into verses, each closing on a line holding a double daṇḍa."
    out, cur = [], []
    for i,l in enumerate(lines):
        cur.append(i)
        if closes(l): out.append(cur); cur = []
    if cur: out.append(cur)
    return out

def closes(l): return '॥' in l or '||' in l

In [ ]:
test_eq(verses(['क ।', 'ख ॥ १ ॥', 'ग', 'घ']), [[0, 1], [2, 3]])

In [ ]:
#| export
def recite_part(ch, ear, lines, kind=None, ref=None, tries=3, max_cer=0.08):
    "Lines rendered in one voice: a `Recitation` and its kind, reference and per-line timings."
    text = ' '.join(word_text(w) for l in lines for w in l.split())
    if kind is None:
        meter = ch.meter([clean(l) for l in lines])
        kind, auto = part_voice(ch, meter)
        ref = ifnone(ref, auto)
    ref = ifnone(ref, 'vasantatilaka' if kind == 'chant' else 'anuṣṭubh')
    ps = pieces(lines, prose=kind=='read')
    gap = ch.gap if kind == 'chant' else ch.prose_gap
    r = best_of(lambda seed: ch.render([piece_text(p) for p in ps], ref, seed, gap), ear, text, tries, max_cer)
    return r, dict(kind=kind, ref=ref, cer=round(r.cer, 3), lines=line_times(len(lines), ps, ear.words(r, [[w for _,w in p] for p in ps])))

In [ ]:
#| export
def recite_section(ch, ear, lines, kind=None, ref=None, parts=None, gap=0.8, **kw):
    "One section's lines to a `Recitation` and its parts' kinds, references, error rates and per-line timings."
    if parts: groups = [(p['lines'], *part_voice(ch, p.get('meter'))) for p in parts]
    else: groups = [(g, kind, ref) for g in ([list(range(len(lines)))] if kind else verses(lines))]
    groups = [o for o in groups if any(word_text(w) for i in o[0] for w in lines[i].split())]
    recs, parts, out, pos = [], [], [None]*len(lines), 0
    for g, k, rf in groups:
        r, info = recite_part(ch, ear, [lines[i] for i in g], k, rf, **kw)
        for i,d in zip(g, shift(info, pos)['lines']): out[i] = d
        parts.append(dict(kind=info['kind'], ref=info['ref'], cer=info['cer'], lines=g))
        recs.append(r); pos += r.ms + round(gap*1000)
    r = concat(recs, gap)
    r.cer = max(p['cer'] for p in parts)
    kinds = {p['kind'] for p in parts}
    return r, dict(kind=kinds.pop() if len(kinds) == 1 else 'mixed', ref=parts[0]['ref'], cer=r.cer, parts=parts, lines=out)

In [ ]:
#| export
def shift(info, off):
    "Move a section's timings `off` ms later."
    for d in info['lines']:
        if not d: continue
        d['s'] += off; d['e'] += off
        for w in d['w']: w[1] += off; w[2] += off
    return info

A stand-in chanter and ear check the bookkeeping without the models: a verse that scans is chanted, the prose after it is read, and the second part's timings start after the first part and the gap.

In [ ]:
from vachaka.core import Recitation
class FakeCh:
    gap, prose_gap = 0.5, 0.3
    def meter(self, lines): return 'anushtubh' if len(lines) == 2 else ''
    def bank_ref(self, name): return {'anuṣṭubh': 'anuṣṭubh', 'mālinī': 'mālinī'}.get(name)
    def render(self, ps, ref, seed, gap): return Recitation(np.zeros(len(ps)*24000, np.float32), [(i*1000, i*1000+900) for i in range(len(ps))], ps)
class FakeEar:
    def hear(self, audio, sr): return ''
    def words(self, rec, ws): return [[[w, s, e] for w in p] for (s,e),p in zip(rec.spans, ws)]
r, info = recite_section(FakeCh(), FakeEar(), ['धर्मक्षेत्रे कुरुक्षेत्रे समवेता युयुत्सवः ।', 'मामकाः पाण्डवाश्चैव किमकुर्वत सञ्जय ॥ १ ॥', 'इति प्रश्नः ॥'])
test_eq([p['kind'] for p in info['parts']], ['chant', 'read'])
test_eq(info['kind'], 'mixed')
test_eq(info['lines'][2]['s'], 2000 + 800)           # two 1 s pieces, then the gap
_, info = recite_section(FakeCh(), FakeEar(), ['क ख ग', 'घ ङ च ॥', 'छ ज झ ।'])
test_eq([p['kind'] for p in info['parts']], ['chant', 'read'])  # the line after the verse is its own part, and scans as nothing
ps = [dict(lines=[0, 1], meter='mālinī'), dict(lines=[2], meter='āryā'), dict(lines=[3], meter='śikhariṇī'), dict(lines=[4], meter=None)]
_, info = recite_section(FakeCh(), FakeEar(), ['क ।', 'ख ॥', 'ग ॥', 'घ ॥', 'ङ ।'], parts=ps)
test_eq([(p['kind'], p['ref']) for p in info['parts']], [('chant', 'mālinī'), ('read', 'anuṣṭubh'), ('chant', 'vasantatilaka'), ('read', 'anuṣṭubh')])

In [ ]:
#| export
def recite(ch, ear, sections, gap=1.2, **kw):
    "Sections (each `dict(lines=…, kind=…, ref=…)`) to one `Recitation` and per-section timings."
    recs, infos, pos = [], [], 0
    for sec in sections:
        r, info = recite_section(ch, ear, sec['lines'], sec.get('kind'), sec.get('ref'), sec.get('parts'), **kw)
        infos.append(shift(info, pos)); recs.append(r); pos += r.ms + round(gap*1000)
    return concat(recs, gap), infos

## CLI

`vachaka-recite jobs.jsonl results.jsonl` renders each job, `dict(id, out, sections)`, to `out` (mp3 or wav) and appends `dict(id, out, ms, sections)`
to the results. A job already in the results is skipped, so a stopped run resumes.

In [ ]:
#| export
@call_parse
def recite_cli(jobs:str, results:str, voice:str='steer', tries:int=3, max_cer:float=0.08, fx:str='warm', pace:float=1.15, pitch:float=1.0):
    "Render a jsonl of jobs, appending timings to a jsonl of results: at `pace`, `pitch` semitones up, with the sound treatment `fx` (core.FX; 'none' for the raw render)."
    from vachaka.check import Ear
    done = {json.loads(l)['id'] for l in open(results)} if Path(results).exists() else set()
    todo = [j for j in map(json.loads, open(jobs)) if j['id'] not in done]
    if not todo: return
    ch, ear = Chanter(voice, pace=pace), Ear()
    for j in todo:
        rec, infos = recite(ch, ear, j['sections'], tries=tries, max_cer=max_cer)
        rec.save(j['out'], fx=None if fx == 'none' else fx, pitch=pitch)
        with open(results, 'a') as f: f.write(json.dumps(dict(id=j['id'], out=j['out'], ms=rec.ms, sections=infos), ensure_ascii=False)+'\n')
        print(j['id'], rec.ms, [i['cer'] for i in infos], flush=True)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()